# 05 - Full Combined Pipeline End-to-End Demo

Run the complete Text-to-Network Engine on both policy + Yelp data in a single notebook, suitable for client presentation.

In [ ]:
import sys, time
sys.path.insert(0, '..')
t0 = time.time()

## Stage 1: Data Ingestion (Policy + Yelp)

In [ ]:
from src.ingest.pdf_reader import read_policy_pdf, pages_to_document
from src.ingest.yelp_reader import load_yelp_reviews

YELP_SAMPLE = 500  # adjust this to scale up

pages = read_policy_pdf()
print(f'Ingested {len(pages)} pages from the climate policy document')

yelp_df = load_yelp_reviews(category_filter='Restaurants', sample_n=YELP_SAMPLE)
print(f'Ingested {len(yelp_df)} Yelp reviews')

## Stage 2: Preprocessing (Both Sources)

In [ ]:
from src.preprocessing.cleaner import clean_text
from src.preprocessing.tokeniser import SpacyTokeniser

tokeniser = SpacyTokeniser()
documents = []

# Policy
cleaned_policy = clean_text(pages_to_document(pages), source='policy')
documents.append(tokeniser.process(cleaned_policy, doc_id='policy_doc', source='policy'))
print(f'Policy: {len(documents[0].sentences)} sentences')

# Yelp
yelp_texts = [(row['review_id'], row['text']) for _, row in yelp_df.iterrows()]
cleaned_yelp = [(did, clean_text(t, source='yelp')) for did, t in yelp_texts]
cleaned_yelp = [(did, t) for did, t in cleaned_yelp if t.strip()]
documents.extend(tokeniser.process_batch(cleaned_yelp, source='yelp'))

total_sents = sum(len(d.sentences) for d in documents)
print(f'Combined: {len(documents)} documents, {total_sents} sentences')

## Stage 3: Concept and Relationship Extraction

In [ ]:
from src.extraction.concept_extractor import ConceptExtractor
from src.extraction.relationship_extractor import RelationshipExtractor

concepts = ConceptExtractor().extract(documents)
relationships = RelationshipExtractor().extract(documents, concepts)
print(f'Extracted {len(concepts)} concepts and {len(relationships)} relationships')

print(f'  Policy-only: {sum(1 for c in concepts if c.source_type == "policy")}')
print(f'  Yelp-only:   {sum(1 for c in concepts if c.source_type == "yelp")}')
print(f'  Shared:      {sum(1 for c in concepts if c.source_type == "both")}')

## Stage 4: Network Construction

In [ ]:
from src.network.graph_builder import GraphBuilder

G = GraphBuilder(min_edge_weight=2).build(concepts, relationships)
print(GraphBuilder.summary(G))

## Stage 5: Network Analysis + Cross-Source Insights

In [ ]:
from src.network.graph_analysis import GraphAnalyser

analyser = GraphAnalyser(G)
centrality_df = analyser.all_centralities()

try:
    partition = analyser.detect_communities_louvain()
except ImportError:
    partition = analyser.detect_communities_label_propagation()

G = analyser.annotate_graph(partition)

print('--- Top 10 Concepts (PageRank) ---')
for _, row in centrality_df.head(10).iterrows():
    print(f"  {row['label']:40s}  PR={row['pagerank']:.4f}  Betw={row['betweenness']:.4f}")

print(f'\nCommunities: {len(set(partition.values()))}')
print(analyser.community_summary(partition).to_string(index=False))

# Cross-source analysis
comparison = analyser.source_comparison_summary()
print(f'\n--- Cross-Source Summary ---')
print(f'  Policy-only: {comparison["policy_only_concepts"]}')
print(f'  Yelp-only:   {comparison["yelp_only_concepts"]}')
print(f'  Shared:      {comparison["shared_concepts"]} ({comparison["overlap_pct"]:.1f}%)')

bridges = analyser.bridging_concepts(top_n=10)
print('\n--- Top Bridging Concepts ---')
for _, row in bridges.head(5).iterrows():
    print(f'  {row["label"]:30s}  score={row["bridge_score"]:.4f}  src={row["source_type"]}')

## Stage 6: Visualisation

In [ ]:
from src.visualisation.static_viz import StaticVisualiser
from src.visualisation.interactive_viz import InteractiveVisualiser

viz = StaticVisualiser('../results')
viz.plot_network_by_source(G, title='Combined Network — Coloured by Source')
viz.plot_source_overlap(G)
viz.plot_network(G, partition=partition, title='Combined Conceptual Network (by Community)')
viz.plot_top_concepts(centrality_df)
viz.plot_community_distribution(partition)
viz.plot_cooccurrence_heatmap(G)

iv = InteractiveVisualiser('../results')
html_path = iv.create_interactive_network(G, partition=partition,
    title='Combined Network Explorer — Policy + Yelp', colour_by_source=True)

print(f'\nPipeline complete in {time.time()-t0:.1f}s')
print(f'Interactive network: {html_path}')

## Key Insights

The combined conceptual network reveals:
- **Central concepts** — the most influential ideas across both policy and review data
- **Thematic communities** — clusters of related concepts forming distinct topic areas
- **Broker concepts** — ideas that bridge different thematic communities
- **Cross-source bridges** — concepts shared between policy and Yelp that connect the two domains
- **Source distribution** — how concepts split between policy-only, yelp-only, and shared